In [1]:
!apt-get -qq update
!apt-get -qq install -y openmpi-bin libopenmpi-dev
!pip install -q mpi4py numba numpy

!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --version | head -n 2

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-cc753a73-75d8-fb4d-bdcf-6bbea1a74faf, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [4]:
GITHUB_USERNAME = "UmirzakovIIDanial"

assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0, (
    "Вы должны указать свой реальный никнейм на GitHub в переменной GITHUB_USERNAME."
)
print(f"Ваш GitHub: {GITHUB_USERNAME}")

Ваш GitHub: UmirzakovIIDanial


In [11]:
%%writefile distributed_cuda_solver.py
import os
import sys
import json
import math
import hashlib
import subprocess
import numpy as np
from mpi4py import MPI
from numba import cuda, float64
import time

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2 # 18
GLOBAL_NX = 512
GLOBAL_NY = 512
STEPS = 400
LAMBDA_DAMP = 0.15
DX = 1.0 / 513.0
DY = 1.0 / 513.0
DT = 2.0e-6

def get_gpu_uuid() -> str:
    out = subprocess.check_output(["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"], encoding="utf-8")
    return out.strip().splitlines()[0].strip()

def derive_student_seed(github_user: str, gpu_uuid: str) -> int:
    digest = hashlib.sha256(f"{github_user.strip()}:{gpu_uuid}".encode("utf-8")).hexdigest()
    return int(digest[:16], 16)

# ================================================================
# TASK 1: Distributed Cartesian Topology & Hardware Context
# ================================================================
def init_distributed_context(github_user: str):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Expected exactly 4 MPI ranks, got {size}.")

    cart_comm = comm.Create_cart(dims=[2, 2], periods=[False, False], reorder=False)
    row_rank, col_rank = cart_comm.Get_coords(rank)
    nbr_up, nbr_down = cart_comm.Shift(0, 1)
    nbr_left, nbr_right = cart_comm.Shift(1, 1)

    cuda.select_device(0)
    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = comm.bcast(gpu_uuid, root=0)
    instance_seed = derive_student_seed(github_user, gpu_uuid)

    return {
        "comm": comm, "cart_comm": cart_comm, "rank": rank,
        "row_rank": row_rank, "col_rank": col_rank,
        "nbr_up": nbr_up, "nbr_down": nbr_down,
        "nbr_left": nbr_left, "nbr_right": nbr_right,
        "gpu_uuid": gpu_uuid, "seed": instance_seed
    }

# ================================================================
# TASK 2: Seeded Heterogeneous Subdomain Initialization
# ================================================================
def initialize_subdomains(ctx):
    ny = nx = 256
    i = np.arange(258)
    j = np.arange(258)

    gy = ctx["row_rank"] * ny + i
    gx = ctx["col_rank"] * nx + j

    y = gy / 513.0
    x = gx / 513.0
    X, Y = np.meshgrid(x, y)

    kappa = 0.16 + 0.08 * np.sin(2 * np.pi * X) * np.cos(2 * np.pi * Y)

    rng = np.random.default_rng(ctx["seed"])
    u = np.zeros((258, 258), dtype=np.float64)
    for _ in range(3):
        cx, cy = rng.uniform(0.2, 0.8, 2)
        amp = rng.uniform(0.5, 1.5)
        sig = 0.05
        u += amp * np.exp(-((X - cx)**2 + (Y - cy)**2) / (2 * sig**2))

    S = 0.5 * np.exp(-((X - 0.5)**2 + (Y - 0.5)**2) / 0.02)

    if ctx["nbr_up"] == MPI.PROC_NULL: u[0, :] = 0.0
    if ctx["nbr_down"] == MPI.PROC_NULL: u[257, :] = 0.0
    if ctx["nbr_left"] == MPI.PROC_NULL: u[:, 0] = 0.0
    if ctx["nbr_right"] == MPI.PROC_NULL: u[:, 257] = 0.0

    return u, kappa, S

# ================================================================
# TASK 3: GPU Halo Pack/Unpack Kernels
# ================================================================
@cuda.jit
def pack_halos_kernel(u, send_up, send_down, send_left, send_right, ny, nx):
    k = cuda.grid(1)
    if k >= ny:
        return
    p = k + 1
    send_up[k]    = u[1,  p]
    send_down[k]  = u[ny, p]
    send_left[k]  = u[p,  1]
    send_right[k] = u[p,  nx]

@cuda.jit
def unpack_halos_kernel(u, recv_up, recv_down, recv_left, recv_right, ny, nx):
    k = cuda.grid(1)
    if k >= ny:
        return
    p = k + 1
    u[0,   p] = recv_up[k]
    u[ny+1,p] = recv_down[k]
    u[p,   0] = recv_left[k]
    u[p,nx+1] = recv_right[k]

# ================================================================
# TASK 4: 2D Shared-Memory Tiled Stencil Kernel
# ================================================================
@cuda.jit
def tiled_stencil_kernel(u_old, u_new, kappa, S, dt, inv_dx2, inv_dy2, lam, r_start, r_end, c_start, c_end):
    smem = cuda.shared.array((18, 18), dtype=float64)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    bx, by = cuda.blockIdx.x, cuda.blockIdx.y

    tid = ty * 16 + tx
    for e in range(tid, 18 * 18, 256):
        sy = e // 18
        sx = e % 18
        smem[sy, sx] = u_old[by * 16 + sy, bx * 16 + sx]

    cuda.syncthreads()

    i = by * 16 + ty + 1
    j = bx * 16 + tx + 1

    if r_start <= i <= r_end and c_start <= j <= c_end:
        c = smem[ty + 1, tx + 1]
        kip = 0.5 * (kappa[i + 1, j] + kappa[i, j])
        kim = 0.5 * (kappa[i, j] + kappa[i - 1, j])
        kjp = 0.5 * (kappa[i, j + 1] + kappa[i, j])
        kjm = 0.5 * (kappa[i, j] + kappa[i, j - 1])

        diff = (kip * (smem[ty + 2, tx + 1] - c) - kim * (c - smem[ty, tx + 1])) * inv_dx2 + \
               (kjp * (smem[ty + 1, tx + 2] - c) - kjm * (c - smem[ty + 1, tx])) * inv_dy2

        u_new[i, j] = c + dt * (diff - lam * c**3 + S[i, j])

# ================================================================
# TASK 5: Dual-Stream Orchestrator & Main Solver
# ================================================================
def run_solver(ctx):
    ny = nx = 256
    u0, kappa, S = initialize_subdomains(ctx)

    u_d = cuda.to_device(u0)
    un_d = cuda.device_array_like(u_d)
    kd = cuda.to_device(kappa)
    Sd = cuda.to_device(S)

    send_h = cuda.pinned_array((4, 256), dtype=np.float64)
    recv_h = cuda.pinned_array((4, 256), dtype=np.float64)
    send_d = cuda.device_array((4, 256), dtype=np.float64)
    recv_d = cuda.device_array((4, 256), dtype=np.float64)

    stream_boundary = cuda.stream()
    stream_interior = cuda.stream()

    grid_pack = (1,)
    block_pack = (256,)
    grid_stencil = (16, 16)
    block_stencil = (16, 16)

    def step_overlapped(u, un):
        # Stage A: Pack & D2H
        pack_halos_kernel[grid_pack, block_pack, stream_boundary](u, send_d[0], send_d[1], send_d[2], send_d[3], ny, nx)
        for k in range(4):
            send_d[k].copy_to_host(send_h[k], stream=stream_boundary)
        stream_boundary.synchronize()

        # Stage B: MPI + Interior Compute
        reqs = []
        if ctx["nbr_down"] != MPI.PROC_NULL: reqs.append(ctx["cart_comm"].Irecv(recv_h[1], source=ctx["nbr_down"], tag=0))
        if ctx["nbr_up"] != MPI.PROC_NULL: reqs.append(ctx["cart_comm"].Irecv(recv_h[0], source=ctx["nbr_up"], tag=1))
        if ctx["nbr_right"] != MPI.PROC_NULL: reqs.append(ctx["cart_comm"].Irecv(recv_h[3], source=ctx["nbr_right"], tag=2))
        if ctx["nbr_left"] != MPI.PROC_NULL: reqs.append(ctx["cart_comm"].Irecv(recv_h[2], source=ctx["nbr_left"], tag=3))

        if ctx["nbr_up"] != MPI.PROC_NULL: reqs.append(ctx["cart_comm"].Isend(send_h[0], dest=ctx["nbr_up"], tag=0))
        if ctx["nbr_down"] != MPI.PROC_NULL: reqs.append(ctx["cart_comm"].Isend(send_h[1], dest=ctx["nbr_down"], tag=1))
        if ctx["nbr_left"] != MPI.PROC_NULL: reqs.append(ctx["cart_comm"].Isend(send_h[2], dest=ctx["nbr_left"], tag=2))
        if ctx["nbr_right"] != MPI.PROC_NULL: reqs.append(ctx["cart_comm"].Isend(send_h[3], dest=ctx["nbr_right"], tag=3))

        # Параллельно считаем внутреннюю часть
        tiled_stencil_kernel[grid_stencil, block_stencil, stream_interior](
            u, un, kd, Sd, DT, 1/DX**2, 1/DY**2, LAMBDA_DAMP, 2, ny-1, 2, nx-1)

        # Stage C: Wait, H2D, Unpack, Boundary Compute
        MPI.Request.Waitall(reqs)
        for k in range(4):
            recv_d[k].copy_to_device(recv_h[k], stream=stream_boundary)

        unpack_halos_kernel[grid_pack, block_pack, stream_boundary](u, recv_d[0], recv_d[1], recv_d[2], recv_d[3], ny, nx)

        tiled_stencil_kernel[grid_stencil, block_stencil, stream_boundary](u, un, kd, Sd, DT, 1/DX**2, 1/DY**2, LAMBDA_DAMP, 1, 1, 1, nx)
        tiled_stencil_kernel[grid_stencil, block_stencil, stream_boundary](u, un, kd, Sd, DT, 1/DX**2, 1/DY**2, LAMBDA_DAMP, ny, ny, 1, nx)
        tiled_stencil_kernel[grid_stencil, block_stencil, stream_boundary](u, un, kd, Sd, DT, 1/DX**2, 1/DY**2, LAMBDA_DAMP, 2, ny-1, 1, 1)
        tiled_stencil_kernel[grid_stencil, block_stencil, stream_boundary](u, un, kd, Sd, DT, 1/DX**2, 1/DY**2, LAMBDA_DAMP, 2, ny-1, nx, nx)

        stream_interior.synchronize()
        stream_boundary.synchronize()

    def step_serialized(u, un):
        tiled_stencil_kernel[grid_stencil, block_stencil](u, un, kd, Sd, DT, 1/DX**2, 1/DY**2, LAMBDA_DAMP, 1, ny, 1, nx)
        pack_halos_kernel[grid_pack, block_pack](u, send_d[0], send_d[1], send_d[2], send_d[3], ny, nx)
        for k in range(4): send_d[k].copy_to_host(send_h[k])

        reqs = []
        if ctx["nbr_down"] != MPI.PROC_NULL: reqs.append(ctx["cart_comm"].Irecv(recv_h[1], source=ctx["nbr_down"], tag=0))
        if ctx["nbr_up"] != MPI.PROC_NULL: reqs.append(ctx["cart_comm"].Irecv(recv_h[0], source=ctx["nbr_up"], tag=1))
        if ctx["nbr_right"] != MPI.PROC_NULL: reqs.append(ctx["cart_comm"].Irecv(recv_h[3], source=ctx["nbr_right"], tag=2))
        if ctx["nbr_left"] != MPI.PROC_NULL: reqs.append(ctx["cart_comm"].Irecv(recv_h[2], source=ctx["nbr_left"], tag=3))

        if ctx["nbr_up"] != MPI.PROC_NULL: reqs.append(ctx["cart_comm"].Isend(send_h[0], dest=ctx["nbr_up"], tag=0))
        if ctx["nbr_down"] != MPI.PROC_NULL: reqs.append(ctx["cart_comm"].Isend(send_h[1], dest=ctx["nbr_down"], tag=1))
        if ctx["nbr_left"] != MPI.PROC_NULL: reqs.append(ctx["cart_comm"].Isend(send_h[2], dest=ctx["nbr_left"], tag=2))
        if ctx["nbr_right"] != MPI.PROC_NULL: reqs.append(ctx["cart_comm"].Isend(send_h[3], dest=ctx["nbr_right"], tag=3))

        MPI.Request.Waitall(reqs)

        for k in range(4): recv_d[k].copy_to_device(recv_h[k])
        unpack_halos_kernel[grid_pack, block_pack](u, recv_d[0], recv_d[1], recv_d[2], recv_d[3], ny, nx)

    # Бенчмарк: Serialized
    t0 = time.perf_counter()
    for _ in range(STEPS):
        step_serialized(u_d, un_d)
        u_d, un_d = un_d, u_d
    serialized_ms = (time.perf_counter() - t0) * 1000

    # Сброс для overlapped
    u0_init, _, _ = initialize_subdomains(ctx)
    u_d = cuda.to_device(u0_init)
    un_d = cuda.device_array_like(u_d)

    energy_trace = []
    l2_trace = []

    def get_metrics(u_dev):
        u_h = u_dev.copy_to_host()
        # Считаем только внутренние ячейки
        local_E = np.sum(u_h[1:257, 1:257]) * DX * DY
        local_L2 = np.sum(u_h[1:257, 1:257]**2) * DX * DY

        # ИСПОЛЬЗУЕМ allreduce (с маленькой буквы)
        E = float(ctx["comm"].allreduce(local_E, op=MPI.SUM))
        L2 = float(ctx["comm"].allreduce(local_L2, op=MPI.SUM))
        return E, L2

    # Бенчмарк: Overlapped
    t0 = time.perf_counter()
    for step in range(STEPS + 1):
        if step % 100 == 0:
            E, L2 = get_metrics(u_d)
            energy_trace.append(E)
            l2_trace.append(L2)
        if step < STEPS:
            step_overlapped(u_d, un_d)
            u_d, un_d = un_d, u_d
    overlapped_ms = (time.perf_counter() - t0) * 1000

    # Проверка Parity (только 1 шаг)
    u_test = cuda.to_device(u0_init)
    un_test = cuda.device_array_like(u_test)
    step_serialized(u_test, un_test)
    u_ser_h = u_test.copy_to_host()

    u_ovl = cuda.to_device(u0_init)
    un_ovl = cuda.device_array_like(u_ovl)
    step_overlapped(u_ovl, un_ovl)
    u_ovl_h = u_ovl.copy_to_host()

    parity_max_diff = float(np.max(np.abs(u_ser_h - u_ovl_h)))

    # PTX Аудит
    ptx = tiled_stencil_kernel.inspect_asm()
    asm = "\n".join(ptx.values())
    ptx_counts = {
        "st_shared_f64": asm.count("st.shared.f64"),
        "ld_shared_f64": asm.count("ld.shared.f64"),
        "bar_sync": asm.count("bar.sync"),
    }
    ptx_audit_passed = all(v > 0 for v in ptx_counts.values())

    manifest = {
        "github_username": sys.argv[1],
        "gpu_uuid": ctx["gpu_uuid"],
        "instance_seed": ctx["seed"],
        "mpi_ranks": 4,
        "cart_dims": [2, 2],
        "serialized_ms": serialized_ms,
        "overlapped_ms": overlapped_ms,
        "parity_max_diff": parity_max_diff,
        "ptx_audit_passed": ptx_audit_passed,
        "ptx_counts": ptx_counts,
        "energy_trace": energy_trace,
        "l2_trace": l2_trace,
        "kappa_max": float(kappa.max()),
        "dt": DT,
        "dx": DX
    }

    if ctx["rank"] == 0:
        with open("submission_manifest.json", "w") as f:
            json.dump(manifest, f, indent=2)
        print(json.dumps(manifest, indent=2))

if __name__ == "__main__":
    if len(sys.argv) < 2:
        sys.exit(1)
    ctx = init_distributed_context(sys.argv[1])
    run_solver(ctx)

Overwriting distributed_cuda_solver.py


In [12]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py",
    GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"MPI execution failed with exit code {proc.returncode}")

{
  "github_username": "UmirzakovIIDanial",
  "gpu_uuid": "GPU-cc753a73-75d8-fb4d-bdcf-6bbea1a74faf",
  "instance_seed": 3966715547620641348,
  "mpi_ranks": 4,
  "cart_dims": [
    2,
    2
  ],
  "serialized_ms": 5951.224598999943,
  "overlapped_ms": 3869.7300769999856,
  "parity_max_diff": 0.0,
  "ptx_audit_passed": true,
  "ptx_counts": {
    "st_shared_f64": 5,
    "ld_shared_f64": 5,
    "bar_sync": 1
  },
  "energy_trace": [
    0.058363162025079046,
    0.05836821858589294,
    0.05837331871792696,
    0.058378420718752,
    0.05838351020993311
  ],
  "l2_trace": [
    0.03884444734845288,
    0.03798362909486131,
    0.037162369893336614,
    0.036377850584337165,
    0.035627526209653475
  ],
  "kappa_max": 0.23999962497115201,
  "dt": 2e-06,
  "dx": 0.001949317738791423
}



### Analytical & Forensic Engineering Responses

**Question 1 (Memory Coalescing & Strided Column Packing):**
In row-major memory, the left and right halo columns `u[1:ny+1, 1]` and `u[1:ny+1, nx]` have a stride of 258 elements (2064 bytes). If a warp of 32 threads attempts to read this column directly, each thread accesses a different memory segment, resulting in 32 separate cache line transactions instead of 1 coalesced transaction. If we attempted an unstrided bounding-box copy, we would transfer 256 elements × 2 columns × 8 bytes = 4096 bytes per rank per step, whereas the actual useful data is only 256 × 8 = 2048 bytes. Thus, an unstrided copy would transfer 2048 extra bytes per boundary. Our `pack_halos_kernel` extracts exactly 2048 bytes into a contiguous 1D buffer, achieving perfect coalescing and minimizing PCIe traffic.

**Question 2 (Shared-Memory Tile Indexing & Bank Conflict Analysis):**
Thread `(tx=0, ty=0)` writes to `smem[1,1]` (bytes 152-159, banks 6-7) and reads from `smem[0,1]`, `smem[2,1]`, `smem[1,0]`, `smem[1,2]`. Thread `(tx=15, ty=15)` writes to `smem[16,16]` (bytes 2432-2439, banks 0-1). In a 64-bit configuration (float64), each element consumes 2 banks (8 bytes). On a Tesla T4 (32 banks of 4 bytes), a warp loading a row (fixed `sy`, varying `sx`) accesses `(2*sx) mod 32` banks. Because 32 threads try to access 64 banks (2 per thread), it results in an unavoidable **2-way bank conflict**. However, loading a column (fixed `sx`, varying `sy`) has a stride of 18 elements (144 bytes = 36 banks). This means threads 0, 8, 16, and 24 will hit the same bank (`(36*sy) mod 32 = (4*sy) mod 32`), resulting in a much worse **4-way bank conflict**.

**Question 3 (PTX Assembly Audit & Warp Synchronization):**
My run reported the following counts in `tiled_stencil_kernel.inspect_asm()`: `st.shared.f64` = 5, `ld.shared.f64` = 5, and `bar.sync` = 1.
If `cuda.syncthreads()` is omitted after loading the shared-memory tile, a memory hazard occurs. Warp 0 might begin evaluating the stencil and reading `smem[ty, tx±1]` while Warp 5 is still writing the halo data into those same shared memory locations. This race condition would cause threads to read stale or uninitialized data, leading to incorrect calculation results, particularly along the boundaries of the 16×16 tiles.

**Question 4 (Multi-Process Context Sharing on a Single Physical GPU):**
When running 4 processes on a single Tesla T4 GPU, the OS and CUDA driver multiplex the 40 SMs and the Copy Engines (DMA) among the four independent CUDA contexts. My measured wall-clock times were `serialized_ms` = 5951.22 ms and `overlapped_ms` = 3869.73 ms (a ~1.54x speedup). The overlap gain is significant but limited compared to four dedicated GPUs connected via InfiniBand. On a single GPU, compute (stencil) and communication (PCIe D2H/H2D) must share the same physical memory bandwidth and DMA engines. On a distributed system with dedicated GPUs, the PCIe bus and network card are not bottlenecked by compute on the same device, allowing for much higher overlap efficiency.

**Question 5 (Point-to-Point Message Deadlock & MPI.PROC_NULL Semantics):**
If a developer replaces non-blocking `Isend/Irecv` with blocking `Send` followed by `Recv` in a fixed order (e.g., all ranks send Up then receive Down), a circular deadlock occurs on large messages. Blocking `Send` does not return until the receiver posts a matching `Recv` (rendezvous protocol, message > buffer threshold ~64KB). Since every rank is stuck waiting for its neighbor to receive, no rank ever reaches its `Recv` call. `MPI.PROC_NULL` simplifies boundary logic by acting as a legal no-op for `Isend/Irecv`. For example, corner rank (0,0) has `nbr_up = PROC_NULL` and `nbr_left = PROC_NULL`. Calling `Isend` on these dummy ranks does nothing and immediately returns, avoiding complex `if` conditionals in the hot loop.

**Question 6 (Explicit CFL Stability & Energy Conservation Proof):**
From the manifest, `kappa_max` = 0.2400. The 2D explicit diffusion stability limit is \(\Delta t_{max} = \Delta x^2 / (4 \cdot \kappa_{max})\). With \(\Delta x = 1/513\), we have \(\Delta x^2 \approx 3.801 \times 10^{-6}\). Thus, \(\Delta t_{max} = 3.801 \times 10^{-6} / 0.96 \approx 3.959 \times 10^{-6}\). The simulation used \(\Delta t = 2.0 \times 10^{-6}\), which safely satisfies \(\Delta t < \Delta t_{max}\).
The Allreduce energy trace is `[0.058363, 0.058368, 0.058373, 0.058378, 0.058383]`. The energy exhibits a slight monotonic *increase* due to the continuous localized heat source \(S(x,y)\) injecting energy into the system. However, the global L2 residual norm trace `[0.03884, 0.03798, 0.03716, 0.03637, 0.03562]` demonstrates **strict monotonic dissipation**, proving that the numerical scheme is stable and damping the nonlinear \(\lambda u^3\) term correctly without numerical blow-up.